# Занятие 30. Практика: переобучение и валидация

Вы **пишете код сами** в пустых ячейках. Блоки **«Легенда»** и **«Дано»** не меняйте.

Теория — занятие 29, ноутбук `Урок_29_Переобучение_и_валидация.ipynb`.
Главная модель: **линейная регрессия** + `PolynomialFeatures` (степень полинома = «сложность формулы»).

### Оценивание (30 баллов)

| № | Тема | Баллы |
|---|------|------:|
| 1 | Импорты и split | 3 |
| 2 | Validation curve по степеням | 5 |
| 3 | Диагноз: under / over / ok | 3 |
| 4 | Learning curve | 4 |
| 5 | K-fold CV | 4 |
| 6 | Параметры vs гиперпараметры | 2 |
| 7 | Утечка в preprocessing | 3 |
| 8 | Выбранная модель | 4 |
| 9 | Итог | 2 |
| | **Итого** | **30** |


---
## Легенда: балансёр RPG «Dragon Gate»

Вы — **балансёр** в команде онлайн-RPG **«Dragon Gate»**.
На финальной арене стоит босс **Алый Дракон**. Урон, который босс наносит игроку, зависит от **силы героя** (`hero_power`): чем сильнее билд, тем выше урон босса по игроку (босс «зеркалит» мощь).

Тестеры прогнали **180 боёв** и записали точки:
- `hero_power` — сила героя в бою (признак x);
- `boss_damage` — урон босса по игроку (цель y).

Точки **шумные**: точность замера, криты, лаги сети. В среднем зависимость похожа на **квадратичную** (y ≈ x²), но шум мешает «угадать формулу на глаз».

### Ваша задача

Подобрать **сложность формулы** — степень полинома (`degree`). Это как крутить ручку «насколько сложный билд»:

| Степень | Метафора | Риск |
|---------|----------|------|
| слишком маленькая | слишком слабый билд | **недообучение** — формула не ловит кривую урона |
| разумная | сбалансированный билд | **норма** — хорошо обобщает на новые бои |
| слишком большая | перекачанный билд | **переобучение** — формула запомнила шум тестов |

**Главная мысль практики:** красивая подгонка под старые бои ≠ честный прогноз на новых. Validation и CV защищают от «перекачанного билда».

### Короткий словарь

| Слово | Значение |
|-------|----------|
| **признак** | `hero_power` — сила героя |
| **цель** | `boss_damage` — урон босса |
| **train** | бои, на которых учим формулу |
| **validation** | бои для сравнения степеней (не для финального хвастовства) |
| **MSE** | средняя квадратичная ошибка: чем меньше, тем ближе прогноз к факту |
| **гиперпараметр** | настройка до обучения (здесь: `degree`) |
| **параметры** | числа, которые модель находит сама (коэффициенты) |


---
## Дано: журнал боёв

Ячейку ниже **не меняйте**. Она создаёт синтетический журнал боёв (как в теории: y ≈ x² + шум), но с игровыми именами колонок.

После запуска у вас будут:
- `df` — таблица с колонками `hero_power` и `boss_damage`;
- `X` — признак (форма `(180, 1)`);
- `y` — цель (форма `(180,)`).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
hero_power = np.linspace(-3, 3, 180)
boss_damage = hero_power ** 2 + rng.normal(0, 1.5, len(hero_power))

df = pd.DataFrame({
    'hero_power': hero_power,
    'boss_damage': boss_damage,
})
X = df[['hero_power']].to_numpy()
y = df['boss_damage'].to_numpy()

print('Боёв в журнале:', len(df))
print(df.head(3).to_string(index=False))

plt.scatter(X[:, 0], y, alpha=0.55, s=28, label='бои (шумные точки)')
xs = np.linspace(X.min(), X.max(), 200)
plt.plot(xs, xs ** 2, color='crimson', linewidth=2, label='истинная кривая ≈ x²')
plt.xlabel('hero_power (сила героя)')
plt.ylabel('boss_damage (урон босса)')
plt.title('Журнал боёв: урон босса vs сила героя')
plt.grid(alpha=0.3)
plt.legend()
plt.show()


---
## Задание 1. Импорты и split — **3 балла**

Подготовьте инструменты для эксперимента баланса.

**Шаг 1.** Импортируйте:
`train_test_split`, `PolynomialFeatures`, `LinearRegression`, `Pipeline`,
`mean_squared_error`, `learning_curve`, `cross_validate`, `KFold`.

**Шаг 2.** Зафиксируйте `RANDOM_STATE`, чтобы все следующие эксперименты сравнивались на **одном и том же** разбиении (иначе «лучшая степень» будет прыгать от запуска к запуску). В эталоне — `42`.

**Шаг 3.** Разделите `X`, `y` на train и validation в пропорции **70 / 30**
(`test_size=0.3`, `random_state=RANDOM_STATE`).
Сохраните `X_train`, `X_val`, `y_train`, `y_val`.

**Шаг 4.** Выведите размеры train и validation.

Train — бои для обучения формулы. Validation — бои, на которых сравниваем степени полинома.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — импортированы все нужные инструменты (включая `Pipeline`).
- **1.0 балл** — задан и используется фиксированный `RANDOM_STATE` (эталон: `42`), чтобы split не менялся между экспериментами.
- **1.0 балл** — получены `X_train`, `X_val`, `y_train`, `y_val` с разбиением 70/30 (~126 / ~54).

### Снижение баллов

- Нет фиксированного `random_state` → минус **0.5**.
- Validation не выделена явно → минус **1.0**.


In [ ]:
from sklearn.model_selection import train_test_split, learning_curve, cross_validate, KFold
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

RANDOM_STATE = 42
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE
)
print('train / val:', len(X_train), len(X_val))


---
## Задание 2. Validation curve по степеням — **5 баллов**

Подберите «сложность формулы» для кривой урона босса.

Для степеней полинома **1–15**:

1. Соберите `Pipeline`: `PolynomialFeatures(degree=..., include_bias=False)` → `LinearRegression`.
2. Обучите (`fit`) **только на train**.
3. Посчитайте **MSE** на train и на validation.
4. Постройте график «степень → MSE» с **двумя** кривыми (train и validation).
5. Сохраните в `best_degree` степень с **минимальным validation MSE**.

График: заголовок, подписи осей, легенда.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — перебраны степени 1–15.
- **1.0 балл** — модель/`Pipeline` обучается только на train (без подглядывания в validation при `fit`).
- **1.0 балл** — для каждой степени посчитан train MSE.
- **1.0 балл** — для каждой степени посчитан validation MSE.
- **1.0 балл** — построен график степень → MSE (две кривые, заголовок, оси, легенда); найден `best_degree`.

### Снижение баллов

- Преобразование/`fit` использует validation → минус **1.0**.
- Нет графика или одной из кривых → минус **0.5**.
- `best_degree` выбран по train MSE → минус **1.0**.


In [ ]:
degrees = list(range(1, 16))
train_scores, val_scores = [], []

for degree in degrees:
    model = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('linreg', LinearRegression()),
    ])
    model.fit(X_train, y_train)
    train_scores.append(mean_squared_error(y_train, model.predict(X_train)))
    val_scores.append(mean_squared_error(y_val, model.predict(X_val)))

plt.plot(degrees, train_scores, marker='o', label='train MSE')
plt.plot(degrees, val_scores, marker='o', label='validation MSE')
plt.xlabel('Степень полинома (сложность формулы)')
plt.ylabel('MSE')
plt.title('Validation curve: сложность билда vs ошибка')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

best_degree = degrees[int(np.argmin(val_scores))]
print('Лучшая степень по validation:', best_degree)
print('train MSE при best:', round(train_scores[best_degree - 1], 3))
print('val MSE при best:', round(val_scores[best_degree - 1], 3))


---
## Задание 3. Диагноз: under / over / ok — **3 балла**

Сравните три «билда»:

1. слабый — степень **1** (недообучение);
2. выбранный — `best_degree` (норма);
3. перекачанный — степень **14** или **15** (переобучение).

Для каждой степени выведите train MSE и validation MSE.

В markdown-ячейке ниже напишите диагноз: где **underfitting**, где **overfitting**, где **ok**.
Опирайтесь на разрыв train / validation и на уровень обеих ошибок (не только на одно число).

### Подробные критерии (для проверки LLM)

- **1.0 балл** — сравнение включает малую степень (1), `best_degree` и большую (14 или 15).
- **1.0 балл** — выведены train и validation MSE для каждого варианта.
- **1.0 балл** — текстовый диагноз: under / over / ok с опорой на разрыв метрик.

### Снижение баллов

- Диагноз без опоры на числа → минус **0.5**.
- `best_degree` выбран/обсуждается по train MSE → минус **1.0**.


In [ ]:
for d in [1, best_degree, 14]:
    model = Pipeline([
        ('poly', PolynomialFeatures(degree=d, include_bias=False)),
        ('linreg', LinearRegression()),
    ])
    model.fit(X_train, y_train)
    tr = mean_squared_error(y_train, model.predict(X_train))
    va = mean_squared_error(y_val, model.predict(X_val))
    print(f'degree={d}: train MSE={tr:.2f}, val MSE={va:.2f}, gap={va - tr:.2f}')


**Диагноз (пример ответа):**

- **degree=1 — недообучение (underfitting):** и train, и validation MSE высокие — формула слишком простая для кривой x² (слабый билд).
- **`best_degree` — норма (ok):** validation MSE минимальна (или близка к минимуму), разрыв train/validation умеренный — сбалансированный билд.
- **degree=14 — переобучение (overfitting):** train MSE падает, validation растёт, разрыв большой — формула запомнила шум тестов (перекачанный билд).

Лучшую степень выбираем по **validation**, а не по train.


---
## Задание 4. Learning curve — **4 балла**

Проверьте, помогает ли «больше боёв в журнале» выбранному билду.

Для `best_degree` постройте **learning curve** (`learning_curve`):

- модель: `Pipeline` с `PolynomialFeatures(degree=best_degree)` + `LinearRegression`;
- данные: все `X`, `y` (CV сама сделает разбиения);
- `cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)`;
- `scoring='neg_mean_squared_error'`;
- несколько размеров train (`train_sizes`, например `np.linspace(0.15, 1.0, 6)`).

На графике покажите **MSE** (умножьте scores на `-1`, потому что sklearn отдаёт отрицательный MSE).
Заголовок, оси, легенда обязательны.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — learning curve для `best_degree` (через `Pipeline` или эквивалент).
- **1.0 балл** — использованы несколько размеров train.
- **1.0 балл** — показаны train и CV/validation scores (как MSE, не «голый» negative score).
- **1.0 балл** — есть график learning curve с заголовком, осями и легендой.

### Снижение баллов

- Кривая для другой степени без объяснения → минус **0.5**.
- Нет CV-кривой или знак MSE не перевёрнут → минус **0.5**.


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
pipe_best = Pipeline([
    ('poly', PolynomialFeatures(degree=best_degree, include_bias=False)),
    ('linreg', LinearRegression()),
])

sizes, tr, va = learning_curve(
    pipe_best, X, y, cv=cv,
    train_sizes=np.linspace(0.15, 1.0, 6),
    scoring='neg_mean_squared_error',
    shuffle=True, random_state=RANDOM_STATE,
)

plt.plot(sizes, -tr.mean(axis=1), marker='o', label='train MSE')
plt.plot(sizes, -va.mean(axis=1), marker='o', label='CV MSE')
plt.xlabel('Размер train (число боёв)')
plt.ylabel('MSE')
plt.title(f'Learning curve (degree={best_degree})')
plt.grid(alpha=0.3)
plt.legend()
plt.show()


---
## Задание 5. K-fold CV — **4 балла**

Один split validation может «повезти». Проверьте `best_degree` кросс-валидацией.

1. Возьмите тот же `Pipeline` со степенью `best_degree` и тот же `KFold` (5 fold).
2. Вызовите `cross_validate` на `X`, `y` с `scoring='neg_mean_squared_error'` и `return_train_score=True`.
3. Выведите MSE на каждом fold, **среднее** и **std** validation MSE.
4. Сравните среднее CV MSE с validation MSE из задания 2 (одним числом/фразой).

Помните: в `cross_validate` колонка `test_score` — это fold-validation, а не финальный test.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — настроен `KFold` с фиксированным `random_state`.
- **1.0 балл** — выполнен `cross_validate` для модели с `best_degree`.
- **1.0 балл** — посчитаны MSE по fold, среднее и std (знак negative score учтён).
- **1.0 балл** — есть сравнение средней CV-оценки с validation MSE из задания 2.

### Снижение баллов

- Метрика с неправильным знаком без преобразования → минус **0.5**.
- Нет сравнения с validation → минус **0.5**.


In [ ]:
pipe_cv = Pipeline([
    ('poly', PolynomialFeatures(degree=best_degree, include_bias=False)),
    ('linreg', LinearRegression()),
])
result = cross_validate(
    pipe_cv, X, y, cv=cv,
    scoring='neg_mean_squared_error',
    return_train_score=True,
)
cv_mse = -result['test_score']
val_mse_holdout = val_scores[best_degree - 1]

print('CV MSE по fold:', cv_mse.round(3))
print('Средняя CV MSE:', round(cv_mse.mean(), 3))
print('std CV MSE:', round(cv_mse.std(), 3))
print('Holdout validation MSE (задание 2):', round(val_mse_holdout, 3))
print(
    'Разница |CV − holdout|:',
    round(abs(cv_mse.mean() - val_mse_holdout), 3),
)


---
## Задание 6. Параметры vs гиперпараметры — **2 балла**

Ответьте в markdown-ячейке ниже.

Что в этом эксперименте **параметры** модели, а что **гиперпараметры**?
Приведите примеры из ноутбука: коэффициенты регрессии, `degree`, настройки split/CV.

Правило: параметры модель находит сама при `fit`; гиперпараметры задаёте вы **до** обучения и выбираете по validation / CV.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — правильно назван пример параметров модели.
- **1.0 балл** — правильно назван пример гиперпараметров (как минимум `degree`).

### Снижение баллов

- Параметры и гиперпараметры перепутаны → минус **1.0**.


**Ответ:**

- **Параметры** — коэффициенты линейной регрессии (веса при степенях `hero_power` и свободный член). Их находит `fit` по train.
- **Гиперпараметры** — степень полинома `degree` (сложность формулы/билда). Её задаём мы и выбираем по validation / CV.
- Настройки вроде `RANDOM_STATE`, `test_size`, число fold в `KFold` — тоже наши решения протокола, но не «веса» модели; главная ручка сложности здесь — `degree`.


---
## Задание 7. Утечка в preprocessing — **3 балла**

Представьте: перед split вы сделали `StandardScaler.fit_transform` или `SimpleImputer.fit_transform` на **всей** таблице боёв.

В markdown объясните:

1. почему это **утечка данных** (data leakage);
2. какой порядок правильный (`fit` на train, `transform` на validation);
3. как это связано именно с preprocessing (запоминание статистик), а не только с самой регрессией.

Метафора: нельзя «подкручивать» баланс, глядя заранее на тестовые бои арены.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — объяснено, почему `fit_transform` на всей таблице до split даёт утечку.
- **1.0 балл** — описан правильный порядок: `fit` на train, `transform` на validation.
- **1.0 балл** — объяснение связано с preprocessing (среднее/std/медиана и т.п.).

### Снижение баллов

- Нет упоминания validation/test как данных, в которые нельзя подглядывать → минус **0.5**.


**Ответ:**

Нельзя делать `fit_transform` на всей таблице до split: scaler/imputer запомнит статистики (среднее, std, медиану) **уже с учётом validation**. Тогда проверочные бои перестают быть «новыми», и MSE выглядит лучше, чем будет на реальной арене — это **утечка**.

Правильно: `fit` только на train, а validation (и test) обрабатывать через `transform` с уже выученными статистиками. Так preprocessing не подглядывает в ответы будущего.


---
## Задание 8. Выбранная модель — **4 балла**

Зафиксируйте сбалансированный билд:

1. Возьмите `best_degree`.
2. Обучите `Pipeline` **только на train**.
3. Посчитайте MSE на **validation** ещё раз.
4. (По желанию) нарисуйте точки validation и кривую прогноза модели.

Это проверка выбранной модели на validation, а не финальный скрытый test продакшена.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — использован `best_degree` из validation curve.
- **1.0 балл** — `fit` выполнен только на train.
- **1.0 балл** — MSE посчитан на validation.
- **1.0 балл** — результат согласуется с выбором из заданий 2–3 (разумный порядок величины).

### Снижение баллов

- Модель обучена с другой степенью без объяснения → минус **0.5**.
- Validation участвует в обучении преобразования/модели → минус **1.0**.


In [ ]:
final_model = Pipeline([
    ('poly', PolynomialFeatures(degree=best_degree, include_bias=False)),
    ('linreg', LinearRegression()),
])
final_model.fit(X_train, y_train)
final_val_mse = mean_squared_error(y_val, final_model.predict(X_val))
print('Final validation MSE:', round(final_val_mse, 3))
print('Выбранная степень (best_degree):', best_degree)

# кривая прогноза на фоне validation-боёв
xs = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)
plt.scatter(X_val[:, 0], y_val, alpha=0.6, s=28, label='validation бои')
plt.plot(xs, final_model.predict(xs), color='darkorange', linewidth=2,
         label=f'формула degree={best_degree}')
plt.xlabel('hero_power')
plt.ylabel('boss_damage')
plt.title('Выбранная модель на validation')
plt.grid(alpha=0.3)
plt.legend()
plt.show()


---
## Задание 9. Итог — **2 балла**

Сформулируйте **три** коротких вывода по своим графикам и числам:

1. что показала **validation curve**;
2. что можно понять по **learning curve**;
3. почему preprocessing / `PolynomialFeatures` в пайплайне нельзя «настраивать», подглядывая в validation.

Пишите по результатам практики, а не как определения из теории.

### Подробные критерии (для проверки LLM)

- **0.7 балла** — вывод про validation curve.
- **0.7 балла** — вывод про learning curve.
- **0.6 балла** — вывод про честный протокол (без подглядывания в validation).

### Снижение баллов

- Выводы выглядят как общая теория и не связаны с графиками/числами → минус **0.5**.


**Итоговые выводы:**

1. **Validation curve:** слишком маленькая степень недообучает (высокий train и val MSE), слишком большая переобучается (train падает, val растёт). Лучшую «сложность билда» берём по минимуму validation MSE — у нас это `best_degree`.
2. **Learning curve:** при росте числа боёв в train ошибки train и CV обычно сближаются; если разрыв остаётся большим, данных мало или модель всё ещё слишком сложная.
3. **Честный протокол:** `fit` преобразований и модели — только на train (или внутри fold). Иначе validation перестаёт имитировать новые бои, и балансёр увидит завышенное качество.
